# Delivery Time Prediction  Linear Regression

**Mission:** Empower local e-commerce and logistics platforms with intelligent, data-driven arrival estimates that optimize driver scheduling, lower operational overhead, and build long-term customer trust through transparency.

**Problem:**Last-mile logistics consumes a massive 53% of total shipping costs, where inaccurate Estimated Times of Arrivals(ETAs) cause roughly 8% of first-attempt deliveries to fail bleeding an average of $17.20 per failure in wasted fuel and driver idling.This project applies a multivariate regression model to explicit driver ratings, vehicle constraints, and geocoded paths to accurately predict delivery times, optimizing fleet schedules and protecting the 84% of customer retention lost to poor delivery experiences.

**Dataset:** `deliverytime.csv`: 45,593 rows, 11 columns.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [2]:
import sys
print(sys.executable)

/home/janviere-munezero/miniconda3/envs/env/bin/python


In [3]:
df = pd.read_csv("data/deliverytime.csv")
print(df.shape)
df.head()


(45593, 11)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Type_of_order,Type_of_vehicle,Time_taken(min)
0,4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,Snack,motorcycle,24
1,B379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,Snack,scooter,33
2,5D6D,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,Drinks,motorcycle,26
3,7A6A,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,Buffet,motorcycle,21
4,70A2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,Snack,scooter,30


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  object 
 1   Delivery_person_ID           45593 non-null  object 
 2   Delivery_person_Age          45593 non-null  int64  
 3   Delivery_person_Ratings      45593 non-null  float64
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Type_of_order                45593 non-null  object 
 9   Type_of_vehicle              45593 non-null  object 
 10  Time_taken(min)              45593 non-null  int64  
dtypes: float64(5), int64(2), object(4)
memory usage: 3.8+ MB


In [5]:
df.describe()

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Time_taken(min)
count,45593.000000,45593.000000,45593.000000,45593.000000,45593.000000,45593.000000,45593.000000
mean,29.544075,4.632367,17.017729,70.231332,17.465186,70.845702,26.294607
std,5.696793,0.327708,8.185109,22.883647,7.335122,21.118812,9.383806
min,15.000000,1.000000,-30.905562,-88.366217,0.010000,0.010000,10.000000
25%,25.000000,4.600000,12.933284,73.170000,12.988453,73.280000,19.000000
50%,29.000000,4.700000,18.546947,75.898497,18.633934,76.002574,26.000000
75%,34.000000,4.800000,22.728163,78.044095,22.785049,78.107044,32.000000
max,50.000000,6.000000,30.914057,88.433452,31.054057,88.563452,54.000000


In [6]:
print("Total rows:", len(df))
print("Unique ID:", df["ID"].nunique())
print("Unique Delivery_person_ID:", df["Delivery_person_ID"].nunique())

Total rows: 45593
Unique ID: 45451
Unique Delivery_person_ID: 1320


In [7]:
df["Delivery_person_Ratings"].value_counts().sort_index()

Delivery_person_Ratings
1.0      38
2.5      20
2.6      22
2.7      22
2.8      19
2.9      19
3.0       6
3.1      29
3.2      29
3.3      25
3.4      32
3.5     249
3.6     207
3.7     225
3.8     228
3.9     197
4.0    1077
4.1    1430
4.2    1418
4.3    1409
4.4    1361
4.5    3303
4.6    8848
4.7    7142
4.8    7148
4.9    7041
5.0    3996
6.0      53
Name: count, dtype: int64

In [8]:
df = df.drop(columns=["ID", "Delivery_person_ID"])
df["Delivery_person_Ratings"] = df["Delivery_person_Ratings"].clip(upper=5.0)
print(df.shape)
df["Delivery_person_Ratings"].describe()

(45593, 9)


count    45593.000000
mean         4.631204
std          0.324613
min          1.000000
25%          4.600000
50%          4.700000
75%          4.800000
max          5.000000
Name: Delivery_person_Ratings, dtype: float64

In [9]:
geo_cols = [
    "Restaurant_latitude", "Restaurant_longitude",
    "Delivery_location_latitude", "Delivery_location_longitude",
]

negative_counts = (df[geo_cols] < 0).sum()
placeholder_counts = (df[geo_cols] == 0.01).sum()

print("Negative values per column:\n", negative_counts)
print("\nExactly 0.01 per column:\n", placeholder_counts)

Negative values per column:
 Restaurant_latitude            431
Restaurant_longitude           162
Delivery_location_latitude       0
Delivery_location_longitude      0
dtype: int64

Exactly 0.01 per column:
 Restaurant_latitude              0
Restaurant_longitude             0
Delivery_location_latitude     327
Delivery_location_longitude    327
dtype: int64


In [10]:
# Fix restaurant coordinate sign errors
df["Restaurant_latitude"] = df["Restaurant_latitude"].abs()
df["Restaurant_longitude"] = df["Restaurant_longitude"].abs()

# Drop rows with unrecoverable placeholder delivery-location coordinates
before = len(df)
df = df[(df["Delivery_location_latitude"] != 0.01) & (df["Delivery_location_longitude"] != 0.01)]
print(f"Dropped {before - len(df)} rows with placeholder delivery location")

print(df.shape)
df[geo_cols].describe()

Dropped 327 rows with placeholder delivery location
(45266, 9)


,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude
count,45266.000000,45266.000000,45266.000000,45266.000000
mean,17.527280,71.293415,17.591282,71.357417
std,7.208973,20.316559,7.209419,20.315410
min,0.000000,0.000000,0.020000,0.020000
25%,12.935662,73.743606,12.994264,73.773613
50%,18.562450,75.902847,18.643935,76.014212
75%,22.744648,78.049117,22.788060,78.117151
max,30.914057,88.433452,31.054057,88.563452


In [11]:
zero_counts = (df[["Restaurant_latitude", "Restaurant_longitude"]] == 0.0).sum()
print(zero_counts)

Restaurant_latitude     3313
Restaurant_longitude    3313
dtype: int64


In [12]:
zero_mask = (df["Restaurant_latitude"] == 0.0) & (df["Restaurant_longitude"] == 0.0)

print("Type_of_order share -- affected rows:")
print(df.loc[zero_mask, "Type_of_order"].value_counts(normalize=True))
print("\nType_of_order share -- full dataset:")
print(df["Type_of_order"].value_counts(normalize=True))

print("\nType_of_vehicle share -- affected rows:")
print(df.loc[zero_mask, "Type_of_vehicle"].value_counts(normalize=True))
print("\nType_of_vehicle share -- full dataset:")
print(df["Type_of_vehicle"].value_counts(normalize=True))

print("\nOriginal row-index range of affected rows:", df.index[zero_mask].min(), "-", df.index[zero_mask].max())
print("Original row-index range of full dataset:", df.index.min(), "-", df.index.max())

Type_of_order share -- affected rows:
Type_of_order
Meal       0.255660
Buffet     0.254754
Snack      0.253245
Drinks     0.236342
Name: proportion, dtype: float64

Type_of_order share -- full dataset:
Type_of_order
Snack      0.253060
Meal       0.251204
Drinks     0.248045
Buffet     0.247691
Name: proportion, dtype: float64

Type_of_vehicle share -- affected rows:
Type_of_vehicle
motorcycle           0.561425
scooter              0.342590
electric_scooter     0.093873
bicycle              0.002113
Name: proportion, dtype: float64

Type_of_vehicle share -- full dataset:
Type_of_vehicle
motorcycle           0.580038
scooter              0.334998
electric_scooter     0.083484
bicycle              0.001480
Name: proportion, dtype: float64

Original row-index range of affected rows: 33 - 45589
Original row-index range of full dataset: 0 - 45592


In [13]:
before = len(df)
df = df[~zero_mask]
print(f"Dropped {before - len(df)} rows with (0,0) placeholder restaurant location")

print(df.shape)
df[geo_cols].describe()

Dropped 3313 rows with (0,0) placeholder restaurant location
(41953, 9)


,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude
count,41953.000000,41953.000000,41953.000000,41953.000000
mean,18.911397,76.923408,18.975064,76.987074
std,5.467829,3.502910,5.469616,3.503073
min,9.957144,72.768726,9.967144,72.778726
25%,12.986047,73.898520,13.065996,73.940327
50%,19.065838,76.618203,19.123249,76.663622
75%,22.751234,78.368855,22.820040,78.405467
max,30.914057,88.433452,31.054057,88.563452


In [14]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df["distance_km"] = haversine_km(
    df["Restaurant_latitude"], df["Restaurant_longitude"],
    df["Delivery_location_latitude"], df["Delivery_location_longitude"],
)

df["distance_km"].describe()

count    41953.000000
mean         9.720284
std          5.603083
min          1.465067
25%          4.657655
50%          9.193021
75%         13.680920
max         20.969489
Name: distance_km, dtype: float64

In [15]:
df = df.drop(columns=geo_cols)
print(df.shape)
df.head()

(41953, 6)


,Delivery_person_Age,Delivery_person_Ratings,Type_of_order,Type_of_vehicle,Time_taken(min),distance_km
0,37,4.9,Snack,motorcycle,24,3.025149
1,34,4.5,Snack,scooter,33,20.183530
2,23,4.4,Drinks,motorcycle,26,1.552758
3,38,4.7,Buffet,motorcycle,21,7.790401
4,32,4.6,Snack,scooter,30,6.210138
